[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/gamaly/dpgs-analysis-2026/blob/main/04_review_and_ai_disclosure.ipynb)


In [1]:
# ======================================================================================================
#  SETUP (inserted by the build script). In Google Colab this downloads the repository so that the
#  `data/` folder sits next to the notebook; on your own computer it does nothing at all.
# ======================================================================================================
import os, sys

REPO_SLUG = "gamaly/dpgs-analysis-2026"     # <github user>/<repository>; set in scripts/99b_build_public_repo.py
BRANCH = "main"

IN_COLAB = "google.colab" in sys.modules      # Colab imports this module in every session; a local Jupyter does not

if IN_COLAB and not os.path.isdir("data"):
    # Colab starts with an empty working directory, so the data folder is missing. Fetch the whole repository
    # (a few tens of MB): first try a shallow `git clone`, and if that fails download the branch zip instead.
    import subprocess, urllib.request, zipfile
    repo_dir = REPO_SLUG.split("/")[-1]
    try:
        subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH,
                        f"https://github.com/{REPO_SLUG}.git", repo_dir], check=True)
    except Exception as err:
        print("git clone failed (", err, ") -> downloading the zip instead")
        url = f"https://github.com/{REPO_SLUG}/archive/refs/heads/{BRANCH}.zip"
        urllib.request.urlretrieve(url, "repo.zip")
        with zipfile.ZipFile("repo.zip") as z:
            z.extractall(".")                                   # unpacks to <repository>-<branch>/
        repo_dir = f"{repo_dir}-{BRANCH}"
    os.chdir(repo_dir)                                         # from here on, "data/" and "extras/" resolve as they do locally
    print("Colab: working in", os.getcwd())
    # Colab already has pandas, numpy and matplotlib. Install only what is missing (nothing, normally).
    import importlib.util
    missing = [p for p in ("pandas", "numpy", "matplotlib") if importlib.util.find_spec(p) is None]
    if missing:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)
elif not IN_COLAB:
    # Local use: run Jupyter from the repository folder (the one containing this notebook and data/).
    assert os.path.isdir("data"), "Start Jupyter in the repository folder: 'data/' must sit next to this notebook."


# 04. Who reviews merged changes, and how much AI-assisted coding is disclosed

**What this notebook does.** Two questions about the ~277,000 merged pull requests (PRs) in the 190+ high-confidence
repos:
1. **Review.** Before a human-authored PR was merged, was it reviewed by a human, only by an AI reviewer, only by
   non-AI automation, or not at all? (script `13b_review_ai.py`, tables "t7")
2. **Disclosure.** How often does the PR's own text say that Claude helped write it, and in how many DPGs?
   (script `03h_ai_disclosure.py`, tables "t8", AI mentions in PR and commit text are matched by keyword and reported as a rough lower bound; lines that state AI was not used are removed before matching; only the Claude keyword group passed the precision test, after a filter for lines stating non-use)

**Per-PR input.** `data/pr_level.csv` is a compact table made by `the final data-preparation script` from the raw cache.
One row per merged PR: repo, DPG, creation and merge dates, author class, review class, an AI-disclosure flag.
It has **no titles, text, logins or e-mails**; the flags were computed from the text before it was discarded.

**Review classes (t7).** Using reviews submitted on or before the merge, ignoring the PR author's own:
* `human_reviewed`: at least one human reviewer;
* `ai_only`: no human reviewer, at least one AI review agent (CodeRabbit, Copilot reviewer, Greptile, Cursor, Codex
  connector, Gemini Code Assist, Claude, Qodo, ...);
* `automation_only`: no human or AI reviewer, only other bots (github-actions, mergify, ...);
* `unreviewed`: no review at all.
A PR with both AI and automation reviews counts as `ai_only`. Dates: **merge date**. 2026 runs to 29 Oct (partial).

**Caveats (stated in the post).** Reviews are capped at the first 10 per PR (4.7% of PRs hit the cap), which can hide
a late human reviewer and leans slightly towards "AI only". Discussion outside GitHub is invisible.
**Disclosure** is a *lower bound and partly a measure of how much people disclose*, not an adoption rate: only the
`claude` keyword group passed the 80% precision gate (80% after fixing the QGIS "no AI was used" template, lines that state AI was not used are removed before matching),
other tools (Copilot, ChatGPT, Cursor, ...) are not counted. Pooled shares are PR-weighted, so big repos dominate;
the post therefore also gives DPG counts and a median.

In [2]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)
DATA = Path("data")
# Each "Claim Nxxx" cell below stores the number(s) it reproduces with record_result(...); the last cell of the
# notebook prints them all with print_results(). (Nothing is read from or written to disk.)
RESULTS = {}

def record_result(claim_id, description, values=(), ok=None, note=""):
    # values: the numbers the cell reproduced; ok: a yes/no test for claims that are not a number
    RESULTS[claim_id] = {"description": description, "values": [float(v) for v in values], "ok": ok, "note": note}

def print_results():
    # plain summary: claim id, what it is, the numbers (or the yes/no test)
    for cid, r in RESULTS.items():
        shown = ", ".join(f"{v:.6g}" for v in r["values"]) if r["values"] else ("yes" if r["ok"] else "no")
        print(f"{cid}  {r['description']}\n      -> {shown}" + (f"   ({r['note']})" if r["note"] else ""))
    print(f"\n{len(RESULTS)} claims reproduced in this notebook")

## 1. Load the per-PR table and build the two populations

In [3]:
pr = pd.read_csv(DATA / "pr_level.csv")
print(len(pr), "merged PRs in", pr.full_name.nunique(), "repos")

# --- review population (t7): human authors, merged 2020 or later, review class present
rv = pr[pr.review_class.notna()].copy()
rv["y"] = rv.merged_date.str[:4].astype(int)
rv["h"] = rv.y.astype(str) + "H" + np.where(rv.merged_date.str[5:7].astype(int) <= 6, "1", "2")   # half-year label
print("human merged PRs (review population):", len(rv), "| share with >=10 stored reviews:", round(rv.reviews_capped.mean(), 3))

# --- disclosure population (t8): PRs created 2020 or later, authors human or AI agent (bots excluded), dated by CREATION
tx = pr[(pr.created_date >= "2020-01-01") & pr.text_class.isin(["human", "ai_agent"])].copy()
tx["year"] = tx.created_date.str[:4]
tx["half"] = tx.year + "H" + np.where(tx.created_date.str[5:7].astype(int) > 6, "2", "1")
tx["signal"] = tx.ai_disclosure_claude.astype(str) == "True"
print("disclosure population:", len(tx))

276525 merged PRs in 181 repos
human merged PRs (review population): 263998 | share with >=10 stored reviews: 0.047


disclosure population: 275862


## 2. Review by year (t7)

In [4]:
CLS = ["human_reviewed", "ai_only", "automation_only", "unreviewed"]

def review_table(key):
    g = rv.groupby([key, "review_class"]).size().unstack(fill_value=0).reindex(columns=CLS, fill_value=0)
    n = g.sum(axis=1)
    out = (g.div(n, axis=0)).round(4)
    out.insert(0, "merged_human_prs", n)
    return out

by_year = review_table("y")
by_year = by_year[(by_year.index >= 2020) & (by_year.index <= 2026)]
print((by_year.assign(**{c: by_year[c] * 100 for c in CLS})).round(2).to_string())

review_class  merged_human_prs  human_reviewed  ai_only  automation_only  unreviewed
y                                                                                   
2020                     28920           71.58     0.00             0.05       28.37
2021                     29970           68.69     0.00             0.11       31.19
2022                     32662           74.74     0.00             0.71       24.55
2023                     40894           74.90     0.00             0.68       24.42
2024                     41403           75.64     0.04             0.39       23.93
2025                     41585           74.92     2.79             0.39       21.89
2026                     48564           67.38     6.48             1.06       25.08


### Claim N054: share of merged changes reviewed by a human, 2022-2026

In [5]:
hr = by_year.human_reviewed * 100
print(hr.loc[2022:2026].round(1).to_string())
record_result("N054", "human-reviewed share (%) 2022..2026", [hr[y] for y in range(2022, 2027)])

y
2022    74.7
2023    74.9
2024    75.6
2025    74.9
2026    67.4


### Claim N055: share reviewed by nobody, 2022-2026
(The post: "between 22% and 25% since 2022".)

In [6]:
un = by_year.unreviewed * 100
print(un.loc[2022:2026].round(1).to_string())
h2 = review_table("h")
un_h2026 = h2.unreviewed["2026H2"] * 100                   # July to 29 October 2026 (partial)
print("unreviewed share 2026H2 (partial): %.1f%%" % un_h2026)
record_result("N055", "unreviewed share (%) 2022..2026, and 2026H2 (partial)", [un[y] for y in range(2022, 2027)] + [un_h2026])

y
2022    24.6
2023    24.4
2024    23.9
2025    21.9
2026    25.1
unreviewed share 2026H2 (partial): 28.0%


### Claim N056: reviewed only by an AI reviewer, 2024-2026

In [7]:
ao = by_year.ai_only * 100
print(ao.loc[2024:2026].round(2).to_string())
record_result("N056", "AI-only reviewed share (%) 2024, 2025, 2026", [ao[2024], ao[2025], ao[2026]])

y
2024    0.04
2025    2.79
2026    6.48


### Claim N095: review bots took about half of the fall in human review, 2025 to 2026
Changes in shares between 2025 and 2026 (percentage points). Human-reviewed falls; AI-only rises by about half as
much; unreviewed rises by about the other half; automation-only (github-actions and similar) makes up the small rest.
Note: computed from the exact shares. The post's 3.1 for unreviewed is the difference of the *rounded* shares
24.8 and 21.7; the exact difference is 3.0.

In [8]:
dlt = (by_year.loc[2026, CLS] - by_year.loc[2025, CLS]) * 100
print(dlt.round(2).to_string())
record_result("N095", "change 2025->2026 in points: human-reviewed, AI-only, unreviewed",
         [dlt["human_reviewed"], dlt["ai_only"], dlt["unreviewed"], dlt["automation_only"]])

review_class
human_reviewed    -7.54
ai_only            3.69
automation_only    0.67
unreviewed         3.19


### Claims N057 and N058: reviews by CodeRabbit and by GitHub's Copilot reviewer
`review_bots_by_year.csv` lists the ten most active non-human reviewer *accounts* per year (all bots or AI agents; no
people are in this file).

In [9]:
bots = pd.read_csv(DATA / "review_bots_by_year.csv")
cr = bots[bots.reviewer_login == "coderabbitai"].set_index("year").reviews
cp = bots[bots.reviewer_login == "copilot-pull-request-reviewer"].set_index("year").reviews
print("CodeRabbit:", cr.to_dict()); print("Copilot reviewer:", cp.to_dict())

CodeRabbit: {2024: 372, 2025: 6794, 2026: 13735}
Copilot reviewer: {2024: 6, 2025: 740, 2026: 3250}


In [10]:
record_result("N057", "CodeRabbit reviews 2024 and 2026", [cr[2024], cr[2026]])

In [11]:
record_result("N058", "Copilot reviewer reviews 2024 and 2026", [cp[2024], cp[2026]])

### Claim N062: the partial second half of 2026 (human-reviewed 64.4%, unreviewed 27.6%)
2026H2 is July to 29 October only; it shows the lowest human share and an uptick in unreviewed changes (the post
now states the unreviewed share). Reviews may still arrive for the newest PRs.

In [12]:
by_half = review_table("h")
by_half = by_half[by_half.index >= "2024H1"]
print((by_half[CLS] * 100).round(1).assign(n=by_half.merged_human_prs).to_string())
record_result("N062", "2026H2 (partial): human-reviewed share and unreviewed share (%)", [by_half.human_reviewed["2026H2"] * 100, by_half.unreviewed["2026H2"] * 100])

review_class  human_reviewed  ai_only  automation_only  unreviewed      n
h                                                                        
2024H1                  75.0      0.0              0.4        24.6  21023
2024H2                  76.3      0.1              0.4        23.2  20380
2025H1                  73.4      3.6              0.4        22.6  21551
2025H2                  76.5      2.0              0.4        21.2  20034
2026H1                  70.6      5.9              0.8        22.7  26345
2026H2                  63.5      7.1              1.4        28.0  22219


## 3. Is it registry-wide? Repo spread and concentration
Repos with at least 20 merged human PRs in both 2023-24 and 2025-26: how did the human-review share change?

### Claim N059: 93 repos with enough PRs; human review fell by more than 10 points in 23, held in 50, rose in 20

In [13]:
def share_by(lo, hi):
    x = rv[(rv.y >= lo) & (rv.y <= hi)]
    return x.groupby("full_name").review_class.agg(n="size", hr=lambda s: (s == "human_reviewed").mean())

a, b = share_by(2023, 2024), share_by(2025, 2026)
j = a.join(b, lsuffix="_p1", rsuffix="_p2", how="inner")
j = j[(j.n_p1 >= 20) & (j.n_p2 >= 20)]
j["change_pts"] = ((j.hr_p2 - j.hr_p1) * 100).round(1)       # rounded to a tenth of a point, as the original script does
fell, held, rose = int((j.change_pts < -10).sum()), int((j.change_pts.abs() <= 10).sum()), int((j.change_pts > 10).sum())
print(len(j), "repos | fell >10 pts:", fell, "| within +-10:", held, "| rose >10:", rose, "| median change:", j.change_pts.median())
record_result("N059", "repos compared; fell; held; rose", [len(j), fell, held, rose])

93 repos | fell >10 pts: 23 | within +-10: 50 | rose >10: 20 | median change: -0.2


C:\Users\Greg\AppData\Local\Temp\ipykernel_68908\2866306434.py:6: Pandas4Warning: The 'lsuffix' and 'rsuffix' keywords in DataFrame.join are deprecated and will be removed in a future version. Use the 'suffixes' keyword instead.
  j = a.join(b, lsuffix="_p1", rsuffix="_p2", how="inner")


### Claim N060: ten fast-growing repos drive the rise in unreviewed changes
Rank repos by the increase in *unreviewed* merged PRs from 2024 to 2026, where 2026 (ten months) is scaled by 4/3 to
a rough full year. This scaling is only used in this ranking.

In [14]:
unrev = rv.assign(u=rv.review_class == "unreviewed").groupby(["full_name", "y"]).u.sum().unstack(fill_value=0)
n_py = rv.groupby(["full_name", "y"]).size().unstack(fill_value=0)
hs = rv.assign(hr=rv.review_class == "human_reviewed").groupby(["full_name", "y"]).hr.mean().unstack()
inc = (unrev[2026] * 4 / 3 - unrev[2024]).sort_values(ascending=False)
top = pd.DataFrame({"unrev_2024": unrev[2024], "unrev_2026_annualised": (unrev[2026] * 4 / 3).round(1), "increase": inc,
                    "n_2024": n_py[2024], "n_2026": n_py[2026], "human_share_2024": hs[2024].round(3),
                    "human_share_2026": hs[2026].round(3)}).loc[inc.index[:10]]
print(top.to_string())
expected = {"ghost", "chamilo", "penpot", "polis", "libresign", "gdevelop", "openelis", "uwazi", "comparia", "qfield"}
found = {n.split("/")[-1].lower().replace("-", "").replace(":", "") for n in top.index}
named = {e for e in expected if any(e in f for f in found)}
print("names in the post found among the top ten:", sorted(named), "of", len(expected))
record_result("N060", "repos in the top-ten list", [len(top)], ok=(len(named) == 10))

                           unrev_2024  unrev_2026_annualised     increase  n_2024  n_2026  human_share_2024  human_share_2026
full_name                                                                                                                    
TryGhost/Ghost                   1358                 2376.0  1018.000000    1567    4600             0.133             0.302
chamilo/chamilo-lms               482                 1497.3  1015.333333     558    1127             0.030             0.004
penpot/penpot                     846                 1649.3   803.333333    1020    2409             0.171             0.486
compdemocracy/polis                32                  718.7   686.666667      43     574             0.256             0.051
LibreSign/libresign               656                 1328.0   672.000000     688    1130             0.047             0.111
4ian/GDevelop                      61                  624.0   563.000000     702     794             0.913           

### Claim N061: in five of those ten, human review fell sharply
"Sharply" = the human-review share fell by more than 10 percentage points between 2024 and 2026.

In [15]:
drop = (top.human_share_2026 - top.human_share_2024) * 100
sharp = int((drop < -10).sum())
print(drop.round(1).to_string()); print("fell by >10 pts:", sharp)
record_result("N061", "top-ten repos where human review fell by more than 10 points, of ten", [sharp, len(top)])

full_name
TryGhost/Ghost               16.9
chamilo/chamilo-lms          -2.6
penpot/penpot                31.5
compdemocracy/polis         -20.5
LibreSign/libresign           6.4
4ian/GDevelop               -50.9
DIGI-UW/OpenELIS-Global-2    -5.9
huridocs/uwazi              -21.6
betagouv/ComparIA            -1.1
opengisch/QField             -7.3
fell by >10 pts: 3


## 4. Disclosed AI-assisted coding (t8)
Flag = the PR's title or body names Claude ("Claude Code", "generated with Claude", "Co-authored-by: Claude",
bare "Claude"), after dropping lines that state AI was *not* used. Only human-authored PRs carry the signal; PRs by
AI-agent accounts are a separate class. Dated by **creation** date, 2026 to 29 Oct.

In [16]:
def disclosure_table(key):
    out = []
    for k, g in tx.groupby(key):
        h = g[g.text_class == "human"]; a_ = g[g.text_class == "ai_agent"]
        per = h.groupby("dpgid").agg(n=("signal", "size"), s=("signal", "mean"))
        per = per[per.n >= 20]                                   # DPGs with enough PRs for a stable share
        top_repos = h[h.signal].full_name.value_counts()
        out.append({key: k, "merged_human_prs": len(h), "human_with_signal": int(h.signal.sum()),
                    "share_human_signal": h.signal.mean() if len(h) else np.nan,
                    "median_dpg_share_dpgs_ge20_prs": per.s.median() if len(per) else np.nan,
                    "n_dpgs_ge20_prs": len(per), "merged_ai_agent_prs": len(a_),
                    "dpgs_with_disclosed_ai_pr": h[h.signal].dpgid.nunique(),
                    "dpgs_with_ai_agent_pr": a_.dpgid.nunique(),
                    "top6_repo_share_of_signal": top_repos.head(6).sum() / top_repos.sum() if len(top_repos) else np.nan})
    return pd.DataFrame(out).set_index(key)

t8y = disclosure_table("year")
print(t8y.round(4).to_string())

      merged_human_prs  human_with_signal  share_human_signal  median_dpg_share_dpgs_ge20_prs  n_dpgs_ge20_prs  merged_ai_agent_prs  dpgs_with_disclosed_ai_pr  dpgs_with_ai_agent_pr  top6_repo_share_of_signal
year                                                                                                                                                                                                            
2020             30733                  0              0.0000                          0.0000               61                    0                          0                      0                        NaN
2021             31507                  1              0.0000                          0.0000               78                    0                          1                      0                     1.0000
2022             34424                  0              0.0000                          0.0000               82                    0                          0      

### Claim N041: DPGs with at least one disclosed AI-assisted PR, 2025 vs 2026

In [17]:
d25, d26 = t8y.dpgs_with_disclosed_ai_pr["2025"], t8y.dpgs_with_disclosed_ai_pr["2026"]
print(d25, d26)
record_result("N041", "DPGs with a disclosed AI PR in 2025 and 2026", [d25, d26])

20 69


### Claim N042: six repositories account for 56% of the 2026 matches
After the template rule (a matching line repeated word for word in 10 or more PRs of one repository is template text, not a disclosure): Pol.is, Kolibri, Altinn, E-Heza, Ghost and OpenELIS Global. QGIS (592 matches before
a matching line repeated word for word in 10 or more PRs of one repository is template text, not a disclosure, 558 of them template text) drops out of the six. Because 69 DPGs have at least one disclosed PR and 35 repos
have ten or more, the signal is not one project, but it is concentrated.

In [18]:
h26 = tx[(tx.year == "2026") & (tx.text_class == "human") & tx.signal]
vc = h26.full_name.value_counts()
print(vc.head(8).to_string()); print("repos with >=10:", int((vc >= 10).sum()), "| repos with any:", len(vc))
top6 = t8y.top6_repo_share_of_signal["2026"] * 100
print("top-6 share (%):", round(top6, 1))
record_result("N042", "share of 2026 matches in the six biggest repos (%)", [top6])

full_name
compdemocracy/polis            449
learningequality/kolibri       401
Altinn/altinn-studio           249
TIP-Global-Health/eheza-app    249
TryGhost/Ghost                 248
DIGI-UW/OpenELIS-Global-2      159
opengovsg/FormSG               144
penpot/penpot                  114
repos with >=10: 35 | repos with any: 70
top-6 share (%): 55.7


### Claim N043: pooled share of merged changes (6.3% so far in 2026, 0.2% in 2025)
The post also says the *typical* DPG is far lower: the median DPG share is 1.8% (DPGs with at least 20 PRs).

In [19]:
pooled = t8y.share_human_signal["2026"] * 100
med = t8y.median_dpg_share_dpgs_ge20_prs["2026"] * 100
print("pooled 2026:", round(pooled, 2), "% | median DPG:", round(med, 2), "%")
print((t8y.share_human_signal * 100).round(2).to_string())
pooled25 = t8y.share_human_signal["2025"] * 100
record_result("N043", "pooled share of merged human PRs with disclosed Claude use, 2026 and 2025 (%)", [pooled, pooled25])

pooled 2026: 6.34 % | median DPG: 1.76 %
year
2020    0.00
2021    0.00
2022    0.00
2023    0.00
2024    0.00
2025    0.20
2026    6.34


### Claim N093: the typical DPG's share is under 2%
Median, across the DPGs with at least 20 merged human PRs in the year, of each DPG's share with a disclosed AI-assisted PR.

In [20]:
med_share = t8y.median_dpg_share_dpgs_ge20_prs["2026"] * 100
n_ge20 = t8y.n_dpgs_ge20_prs["2026"]
print(round(med_share, 2), "% across", n_ge20, "DPGs")
record_result("N093", "median DPG share (%) in 2026; DPGs with >= 20 PRs", [med_share, n_ge20])

1.76 % across 86 DPGs


### Claim N094: 20 DPGs in 2025 with 86 disclosed PRs; 69 DPGs in 2026 with 3,152
The unit is a merged human PR whose title or body mentions Claude (after the negation filter and the template rule).

In [21]:
record_result("N094", "disclosed AI-assisted merged PRs in 2025 and 2026", [t8y.human_with_signal["2025"], t8y.human_with_signal["2026"]])
print(t8y.human_with_signal[["2025", "2026"]].to_dict())

{'2025': 86, '2026': 3152}


### Claim N090: precision of the Claude-mention signal, 24 of 30 (round 2)
Hand check of 30 randomly drawn matches (seed 2030) after the negation fix (lines that state AI was not used are removed before matching): a match is correct when the text
says AI helped write the change. `pr_text_ai_precision.csv` keeps only id, group, round and verdict (1 = correct);
the matched text itself is not released. The 95% interval on 80% is roughly 62-91%.

In [22]:
pg = pd.read_csv(DATA / "pr_text_ai_precision.csv")
r2 = pg[(pg.group == "claude") & (pg["round"] == 2)]
ok, n = int(r2.verdict.sum()), len(r2)
print(pg.groupby(["round", "group"]).verdict.agg(["sum", "count", "mean"]).round(2).to_string())
print(f"round 2, claude: {ok} of {n} = {ok / n:.0%}")
record_result("N090", "round-2 Claude precision: correct, checked, percent", [ok, n, ok / n * 100])

                     sum  count  mean
round group                          
1     chatgpt_codex   10     30  0.33
      claude          25     30  0.83
      copilot         16     28  0.57
      cursor           0      1  0.00
      generic         20     30  0.67
      other_tool       5     30  0.17
2     claude          24     30  0.80
round 2, claude: 24 of 30 = 80%


### Claim N091: QGIS's pull-request template line was removed by the template rule (558 PRs)
QGIS's template asks authors to say whether AI tools ("Copilot, Claude, or something similar") were used and says
that pull requests with AI-written code are currently not accepted while the project evaluates its policy.
The same line is in every PR, ticked or not (the text export loses the tick), so it is not a disclosure. The a matching line repeated word for word in 10 or more PRs of one repository is template text, not a disclosure
rule drops a matching line repeated word for word in 10 or more PRs of one repo. `disclosure_template_dropped.csv`
counts the PRs dropped per repo (computed by the build script from the raw cache). Before that, the older negation
filter (lines that state AI was not used are removed before matching) had removed 129 "certify that no AI tool was used" lines, also all in QGIS.

In [23]:
td = pd.read_csv(DATA / "disclosure_template_dropped.csv").set_index("full_name").prs_dropped_as_template
qc = pd.read_csv(DATA / "qgis_template_check.csv").iloc[0]
print(td.sort_values(ascending=False).head(6).to_dict())
print("QGIS lines removed earlier by the negation filter:", int(qc.n_qgis_lines_removed_by_negation_filter),
      "| template wording about evaluating AI use present in cached QGIS PR bodies:", qc.qgis_template_text_in_cached_pr_body)
record_result("N091", "QGIS PRs dropped as template text", [td["qgis/QGIS"]])

{'qgis/QGIS': 558, 'OpenFn/lightning': 250, 'opensupplyhub/open-supply-hub': 60, 'nextcloud/server': 28, '4ian/GDevelop': 3, 'Aam-Digital/ndb-core': 2}
QGIS lines removed earlier by the negation filter: 129 | template wording about evaluating AI use present in cached QGIS PR bodies: True


## 6. More claims (v7)

### Claim N108: three repositories hold much of the machine-only review
Ghost, Altinn and OpenELIS have the most AI-only reviews in 2026. Without them, how far does the human-reviewed share fall from
2025 to 2026, against the fall for all repositories (percentage points)?

In [24]:
THREE = ("tryghost/ghost", "altinn/altinn-studio", "openelis-global-2")           # lower-case repository names (OpenELIS: any owner)
is_three = rv.full_name.str.lower().apply(lambda n: n.endswith(THREE))
print("repos excluded:", sorted(rv.loc[is_three, "full_name"].unique()))
def human_share(x, y):
    return (x[x.y == y].review_class == "human_reviewed").mean() * 100
without = rv[~is_three]
h25, h26 = human_share(without, 2025), human_share(without, 2026)
all25, all26 = human_share(rv, 2025), human_share(rv, 2026)
print(f"without the three: {h25:.1f}% -> {h26:.1f}% ({h26 - h25:+.1f} points) | all repos: {all25:.1f}% -> {all26:.1f}% ({all26 - all25:+.1f} points)")
record_result("N108", "human-reviewed share (%) without the three repos, 2025 and 2026; change in points; change for all repos",
         [h25, h26, h26 - h25, all26 - all25, abs(h26 - h25) / abs(all26 - all25)])

repos excluded: ['Altinn/altinn-studio', 'DIGI-UW/OpenELIS-Global-2', 'TryGhost/Ghost']
without the three: 77.8% -> 72.7% (-5.2 points) | all repos: 74.9% -> 67.4% (-7.5 points)


## 5. Summary of the numbers reproduced here

In [25]:
print_results()

N054  human-reviewed share (%) 2022..2026
      -> 74.74, 74.9, 75.64, 74.92, 67.38
N055  unreviewed share (%) 2022..2026, and 2026H2 (partial)
      -> 24.55, 24.42, 23.93, 21.89, 25.08, 27.95
N056  AI-only reviewed share (%) 2024, 2025, 2026
      -> 0.04, 2.79, 6.48
N095  change 2025->2026 in points: human-reviewed, AI-only, unreviewed
      -> -7.54, 3.69, 3.19, 0.67
N057  CodeRabbit reviews 2024 and 2026
      -> 372, 13735
N058  Copilot reviewer reviews 2024 and 2026
      -> 6, 3250
N062  2026H2 (partial): human-reviewed share and unreviewed share (%)
      -> 63.52, 27.95
N059  repos compared; fell; held; rose
      -> 93, 23, 50, 20
N060  repos in the top-ten list
      -> 10
N061  top-ten repos where human review fell by more than 10 points, of ten
      -> 3, 10
N041  DPGs with a disclosed AI PR in 2025 and 2026
      -> 20, 69
N042  share of 2026 matches in the six biggest repos (%)
      -> 55.6789
N043  pooled share of merged human PRs with disclosed Claude use, 2026 and 